# MMDP-OD-RTDP Main Report

This notebook serves as the master empirical report for comparing **Baseline Real-Time Dynamic Programming (RTDP)** against **Operator Decomposition (OD) RTDP** in stochastic Multi-Agent Pathfinding environments (MMDP).

It contains two major sections:
1. **Interactive Visualization:** Dynamically solving a map and stepping through the resulting multi-agent policy.
2. **Empirical Benchmarking:** Running head-to-head algorithm comparisons to demonstrate OD RTDP's massive computational advantages.

### 1. Repository Setup & Module Import
The project is structured as a highly optimized modular Python framework. The code below checks if it is running in Google Colab and fetches the repository if necessary.

In [ ]:
import sys
import os
from pathlib import Path
import time
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

REPO_URL = "https://github.com/t-rays/MMDP-OD-RTDP-PROJECT.git"
REPO_NAME = "MMDP-OD-RTDP-PROJECT"

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    sys.path.append('src')
else:
    src_path = str(Path('.').resolve() / 'src')
    if src_path not in sys.path:
        sys.path.append(src_path)
        
from grid_mmdp import GridMMDP, MMDPConfig
from map_creator import create_map_instance
from heuristic import ShortestPathHeuristic
from baseline_rtdp import BaselineRTDP, RTDPConfig
from od_rtdp import OperatorDecompositionRTDP
from evaluation import evaluate_policy, EvaluationConfig
from resource_monitor import ResourceMonitor
from visualizer import TrajectoryVisualizer

print("✅ Framework and Visualizers loaded successfully.")

## Part I: Interactive Grid Visualizer
Here we instantiate the `GridMMDP` environment, solve it rapidly using Operator Decomposition RTDP, and use our `TrajectoryVisualizer` widget to watch the agents execute the resulting policy. 

*Use the Play button or Slider to scrub through the steps!*

In [ ]:
# Load Map and Environment
map_instance = create_map_instance('maps/diagnostic/crossing-9-9', scenario_number=1, task_offset=0, n_agents=3)
mdp = GridMMDP(map_instance, MMDPConfig(slip_to_stay_probability=0.20))
heuristic = ShortestPathHeuristic(mdp)

# Solve with OD RTDP (2 seconds)
print(f"Solving {map_instance.grid_map.name} with OD RTDP...")
planner = OperatorDecompositionRTDP(mdp, heuristic, RTDPConfig(time_limit_seconds=2.0))
planner.solve()

# Evaluate and Capture Trajectories
eval_summary = evaluate_policy(mdp, planner, EvaluationConfig(episodes=1, seed=42))

# Launch Interactive Visualizer
viz = TrajectoryVisualizer(mdp, planner)
viz.show_with_tree()

## Part II: Empirical Benchmarking
We define a reusable function that runs both Baseline and OD RTDP on a given map under a strict time limit. This demonstrates how OD RTDP's reduced branching factor allows for exponentially more Bellman backups.

In [ ]:
def run_comparison(map_path: str, n_agents: int, time_limit: float = 5.0, episodes: int = 50):
    map_instance = create_map_instance(map_path, scenario_number=1, task_offset=0, n_agents=n_agents)
    mdp = GridMMDP(map_instance, MMDPConfig(slip_to_stay_probability=0.20))
    heuristic = ShortestPathHeuristic(mdp)
    
    print(f"\n{'='*60}\nRunning Map: {map_instance.grid_map.name} ({n_agents} Agents) | Time Limit: {time_limit}s\n{'='*60}")
    
    config = RTDPConfig(time_limit_seconds=time_limit, seed=42)
    eval_config = EvaluationConfig(episodes=episodes, seed=101)
    
    # Baseline RTDP
    baseline_planner = BaselineRTDP(mdp, heuristic, config)
    print("Running Baseline RTDP...")
    with ResourceMonitor() as monitor:
        baseline_result = baseline_planner.solve()
    baseline_mem = monitor.snapshot().peak_rss_delta_mb
    baseline_eval = evaluate_policy(mdp, baseline_planner, eval_config)
    
    # OD RTDP
    od_planner = OperatorDecompositionRTDP(mdp, heuristic, config)
    print("Running OD RTDP...")
    with ResourceMonitor() as monitor:
        od_result = od_planner.solve()
    od_mem = monitor.snapshot().peak_rss_delta_mb
    od_eval = evaluate_policy(mdp, od_planner, eval_config)
    
    return {
        "map": map_instance.grid_map.name,
        "agents": n_agents,
        "baseline": {
            "trials": baseline_result.trials_completed,
            "backups": baseline_result.bellman_backups,
            "success": baseline_eval.summary.success_rate,
            "steps": baseline_eval.summary.mean_steps_successful_episodes,
            "memory_mb": baseline_mem
        },
        "od": {
            "trials": od_result.trials_completed,
            "backups": od_result.bellman_backups,
            "success": od_eval.summary.success_rate,
            "steps": od_eval.summary.mean_steps_successful_episodes,
            "memory_mb": od_mem
        }
    }

In [ ]:
results = []

# Run 1: Simple Empty 8x8 Map (3 Agents)
results.append(run_comparison('maps/empty-8-8', n_agents=3, time_limit=5.0))

# Run 2: Diagnostic Crossing 9x9 Map (3 Agents)
results.append(run_comparison('maps/diagnostic/crossing-9-9', n_agents=3, time_limit=5.0))

# Run 3: Hard Warehouse Map (3 Agents) - Allow slightly more time due to maze complexity
results.append(run_comparison('maps/warehouse-10-20-10-2-1', n_agents=3, time_limit=10.0))

print("\n✅ All Experiments completed.")

### Empirical Comparison Results (Tabular)


In [ ]:
def generate_markdown_table(results):
    table = "| Map | Agents | Algorithm | Trials | Bellman Backups | Success Rate | Peak RAM (MB) |\n"
    table += "|---|---|---|---|---|---|---|\n"
    
    for res in results:
        b = res['baseline']
        table += f"| **{res['map']}** | {res['agents']} | **Baseline RTDP** | {b['trials']:,} | {b['backups']:,} | {b['success']*100:.1f}% | {b['memory_mb']:.1f} MB |\n"
        
        o = res['od']
        table += f"| | | **OD RTDP** | {o['trials']:,} | {o['backups']:,} | {o['success']*100:.1f}% | {o['memory_mb']:.1f} MB |\n"
    
    return table

display(Markdown(generate_markdown_table(results)))

### Data Visualization (Analytics Charts)
To make the structural advantages of Operator Decomposition perfectly clear, we plot the **Bellman Backups (Logarithmic Scale)** and the **Evaluation Success Rate** side-by-side.

In [ ]:
maps = [r['map'] for r in results]
base_backups = [r['baseline']['backups'] for r in results]
od_backups = [r['od']['backups'] for r in results]
base_succ = [r['baseline']['success']*100 for r in results]
od_succ = [r['od']['success']*100 for r in results]

x = np.arange(len(maps))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Bellman Backups (Log Scale)
ax1.bar(x - width/2, base_backups, width, label='Baseline', color='#e74c3c')
ax1.bar(x + width/2, od_backups, width, label='OD RTDP', color='#2ecc71')
ax1.set_ylabel('Total Bellman Backups (Log Scale)')
ax1.set_title('Computational Throughput (Bellman Backups)')
ax1.set_xticks(x)
ax1.set_xticklabels(maps, rotation=15)
ax1.set_yscale('log')
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Plot 2: Success Rate
ax2.bar(x - width/2, base_succ, width, label='Baseline', color='#e74c3c')
ax2.bar(x + width/2, od_succ, width, label='OD RTDP', color='#2ecc71')
ax2.set_ylabel('Success Rate (%)')
ax2.set_title('Policy Robustness (Success Rate over 50 Episodes)')
ax2.set_xticks(x)
ax2.set_xticklabels(maps, rotation=15)
ax2.set_ylim(0, 110)
ax2.legend()
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Conclusion

### The Curse of Dimensionality
Baseline RTDP struggles because the branching factor of the joint-action space grows exponentially ($|A|^n$). With 3 agents having 5 moves each, the branching factor is 125. This causes extreme memory pressure (as seen in the RAM column) and slows down state expansion, severely limiting the number of Bellman backups that can be completed in 5 seconds.

### The OD Solution
Operator Decomposition completely mitigates this by breaking simultaneous joint actions into a sequence of individual decisions. The branching factor drops to $|A| = 5$. As a result, the state-space tree is much narrower (though deeper), allowing the planner to compute vastly more Bellman backups using a fraction of the RAM. 

As proven in the charts above, this structural advantage consistently yields highly robust policies (approaching 100% success rates) even on complex maps like the Warehouse.